In [ ]:
import pandas as pd
import numpy as np

# preprocessing
from sklearn.model_selection import train_test_split,RandomizedSearchCV
from sklearn.preprocessing import StandardScaler,OneHotEncoder
from imblearn.over_sampling import RandomOverSampler

# model machine learning
from sklearn.compose import ColumnTransformer
# from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,log_loss
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [2]:
df = pd.read_csv('../0.dataset/dataset_Heart_attack_clean.csv')
df_x = df.drop(columns='heart_attack')
df_y = df[['heart_attack']]

# 1.Binary Classification Logistic Regeresion

In [3]:
feature_numerik = ['age','body_mass_index','systolic_blood_pressure','forced_expiratory_volume_1','time_to_event_or_censoring']
feature_categori = ['gender']

preprocessor = ColumnTransformer(
    transformers=[
        ('numerik_scaler',StandardScaler(),feature_numerik),
        ('categori_encoding',OneHotEncoder(),feature_categori)
        ],
   remainder='passthrough' # Kolom kategori yang sudah ter-encode dari sananya akan aman lewat lewat sini
)

In [4]:
selector_model = LogisticRegression(solver='liblinear',random_state=42)
model_LogisticRegression = Pipeline([
    ('preprocessing',preprocessor),
    ('oversampling',RandomOverSampler(random_state=42)),
    ('feature_selection',SequentialFeatureSelector(estimator=selector_model,direction='forward',cv=3)),
    ('model_classifier',LogisticRegression(solver='liblinear',random_state=42))
])

In [5]:
params = {
    'feature_selection__n_features_to_select': [13, 14,'auto',5,9,7],
    'model_classifier__C': np.logspace(-5, -2, 10),
    'model_classifier__penalty': ['l2'],
    'model_classifier__max_iter': [500, 1000, 2000],
    'model_classifier__solver': ['lbfgs','liblinear']
}
X_train,X_test,y_train,y_test = train_test_split(df_x,df_y,test_size=0.2,random_state=42)

random_search = RandomizedSearchCV(estimator=model_LogisticRegression,param_distributions=params,n_iter=10,cv=5,scoring='f1',random_state=42,n_jobs=-1,verbose=3)
random_search.fit(X_train,y_train.values.ravel())

Fitting 5 folds for each of 10 candidates, totalling 50 fits


ValueError: 
All the 50 fits failed.
It is very likely that your model is misconfigured.
You can try to debug the error by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
50 fits failed with the following error:
Traceback (most recent call last):
  File "d:\all-packmanager-app\miniconda3\envs\Hafidz29Python\Lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\all-packmanager-app\miniconda3\envs\Hafidz29Python\Lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "d:\all-packmanager-app\miniconda3\envs\Hafidz29Python\Lib\site-packages\sklearn\pipeline.py", line 655, in fit
    Xt = self._fit(X, y, routed_params, raw_params=params)
  File "d:\all-packmanager-app\miniconda3\envs\Haf

In [ ]:
best_model_pipeline = random_search.best_estimator_
preprocessor_step = best_model_pipeline.named_steps['preprocessing']
sfs_step = best_model_pipeline.named_steps['feature_selection']

kolom_setelah_transformasi = preprocessor_step.get_feature_names_out()
fitur_terpilih = kolom_setelah_transformasi[sfs_step.get_support()]

print(f'\nHyperparameter Terbaik:\n{random_search.best_params_}')
print(f'\nFitur Terbaik Yang Terpilih:\n{list(fitur_terpilih)}')

#cek score
tes_accuracy = best_model_pipeline.score(X_test, y_test)

y_pred_test = best_model_pipeline.predict(X_test)
y_prob_test = best_model_pipeline.predict_proba(X_test)

y_pred_train = best_model_pipeline.predict(X_train)
y_prob_train = best_model_pipeline.predict_proba(X_train)

print(f'\nAkurasi Validasi Terbaik: {random_search.best_score_*100:.2f}%')
print(f'\nAkurasi Data Test: {tes_accuracy*100:.2f}%')


Hyperparameter Terbaik:
{'model_classifier__solver': 'lbfgs', 'model_classifier__penalty': 'l2', 'model_classifier__max_iter': 1000, 'model_classifier__C': np.float64(0.002154434690031882), 'feature_selection__n_features_to_select': 5}

Fitur Terbaik Yang Terpilih:
['numerik_scaler__age', 'numerik_scaler__body_mass_index', 'numerik_scaler__systolic_blood_pressure', 'numerik_scaler__forced_expiratory_volume_1', 'numerik_scaler__time_to_event_or_censoring']

Akurasi Validasi Terbaik: 99.37%

Akurasi Data Test: 99.38%


In [ ]:
def evaluate_model(pred,test,prob, evaluate, model_name='Logistic Regression'):
    accuracy = accuracy_score(test,pred)
    precision = precision_score(test,pred)
    recall = recall_score(test, pred)
    f1 = f1_score(test,pred)
    roc_auc = roc_auc_score(test,prob[:, 1])
    logloss = log_loss(test,prob)
    
    data = {
        'Model': [model_name],
        'Evaluated': [evaluate],
        'Accuracy': [accuracy],
        'Precision': [precision],
        'Recall': [recall],
        'F1-Score': [f1],
        'ROC-AUC': [roc_auc],
        'Log Loss': [logloss]
    }
    df_result = pd.DataFrame(data)
    return df_result

In [ ]:
def check_model_fit(df_eval_train, df_eval_test):
    df_combined = pd.concat([df_eval_train, df_eval_test], ignore_index=True)
    acc_train = df_eval_train['Accuracy'].values[0]
    acc_test = df_eval_test['Accuracy'].values[0]
    gap = acc_train - acc_test

    if acc_train < 0.60 and acc_test < 0.60:
        status = 'Underfitting (akurasi rendah)'
    elif gap > 0.05:
        status = f'Overfitting (gap:{gap*100:.2f}%)'
    elif gap < -0.05:
        status = 'Overfitting / Data leakage (Test > Train)'
    else:
        status = 'Good fit'
    df_combined['Status'] = status
    return df_combined

In [ ]:
df_eval_train = evaluate_model(y_pred_train, y_train, y_prob_train, evaluate='Training')
df_eval_test = evaluate_model(y_pred_test, y_test, y_prob_test, evaluate='Test')
df_eval = check_model_fit(df_eval_train, df_eval_test)

print('================================= PREDIKSI DENGAN SAMPLE DATASET ===================================')
print(df_eval.to_string())
print("\n" + "="*100 + "\n")

================================= PREDIKSI DENGAN SAMPLE DATASET ===================================
                 Model Evaluated  Accuracy  Precision  Recall  F1-Score   ROC-AUC  Log Loss    Status
0  Logistic Regression  Training  0.993704        0.0     0.0       0.0  0.804096  0.035478  Good fit
1  Logistic Regression      Test  0.993791        0.0     0.0       0.0  0.837049  0.034644  Good fit




## Melakukan Prediksi Dengan Data Random

In [ ]:
data_4_pasien = {
'gender': ['F', 'F', 'F', 'F'],
    'age': [31, 72, 28, 63],
    'body_mass_index': [25.9, 28.6, 27.0, 27.0],
    'smoker': [0, 0, 0, 0],  # Diasumsikan dari data numerik biner Anda
    'systolic_blood_pressure': [112.0, 125.0, 102.0, 130.0],
    'hypertension_treated': [1, 1, 0, 0],
    'family_history_of_cardiovascular_disease': [0, 0, 0, 0],
    'atrial_fibrillation': [0, 0, 0, 0],
    'chronic_kidney_disease': [0, 1, 0, 0],
    'rheumatoid_arthritis': [0, 0, 0, 0],
    'diabetes': [0, 0, 0, 0],
    'chronic_obstructive_pulmonary_disorder': [0, 0, 0, 0],
    'forced_expiratory_volume_1': [90.96, 88.00, 98.37, 100.00], # Mengisi nilai representatif berdasarkan teks data Anda
    'time_to_event_or_censoring': [9833, 100, 6954, 100],        # Mengisi nilai representatif berdasarkan teks data Anda
    'heart_attack': [1, 1, 1, 0]
}
data_pasien_baru_x = pd.DataFrame(data_4_pasien)
data_pasien_baru_y = data_pasien_baru_x['heart_attack']

In [ ]:
print("=== Melakukan Prediksi Data Pasien Baru ===")
prediksi_pasien = best_model_pipeline.predict(data_pasien_baru_x)
probabilitas_pasien = best_model_pipeline.predict_proba(data_pasien_baru_x)

hasil_df = data_pasien_baru_x.copy()
hasil_df['Prediksi Serangan Jantung'] = prediksi_pasien
hasil_df['Peluang Aman(%)'] = probabilitas_pasien[:,0] * 100
hasil_df['Peluang Terkena (%)'] = probabilitas_pasien[:,1] * 100

akurasi_bawaan = best_model_pipeline.score(data_pasien_baru_x, data_pasien_baru_y)
df_eval_data_baru = evaluate_model(
    pred=prediksi_pasien, 
    test=data_pasien_baru_y, 
    prob=probabilitas_pasien, 
    evaluate='Data_Baru'
)

print(f"Akurasi Model: {akurasi_bawaan * 100:.2f}%")
print("\nTabel Skor Evaluasi Lengkap Data Baru:")
print(df_eval_data_baru.to_string(index=False))
hasil_df[['Prediksi Serangan Jantung', 'Peluang Aman(%)', 'Peluang Terkena (%)']]

=== Melakukan Prediksi Data Pasien Baru ===
Akurasi Model: 25.00%

Tabel Skor Evaluasi Lengkap Data Baru:
              Model Evaluated  Accuracy  Precision  Recall  F1-Score  ROC-AUC  Log Loss
Logistic Regression Data_Baru      0.25        0.0     0.0       0.0 0.333333  4.055837


,Prediksi Serangan Jantung,Peluang Aman(%),Peluang Terkena (%)
0,0,99.689913,0.310087
1,0,98.898068,1.101932
2,0,99.734163,0.265837
3,0,99.091340,0.908660


# 2.Multiclass Classification